## Process output of simulations

In [1]:
import os
import glob
import gzip
import math
import random
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
import shapely.wkt as wkt
from shapely.geometry import Point, LineString, box
from shapely.ops import nearest_points
import lxml.etree as ET
import tqdm
import wandb
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.transforms import LineGraph
import processing_io as pio
import re 
import os
import glob
import math
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import torch
from collections import defaultdict

import processing_io as pio
from torch_geometric.transforms import LineGraph

from torch_geometric.data import Data, Batch
import shapely.wkt as wkt
from tqdm import tqdm
import fiona
import os

import alphashape
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from shapely.geometry import Point
import random

districts = gpd.read_file("../../../../data/visualisation/districts_paris.geojson")

# Parameters to adapt
districts_of_policy_implementation = [1, 2, 3, 4]
# is_for_1pm = True
# plot_in_percentage = True

string_is_for_1pm = "pop_1pm"
string_district_of_interest = "_".join([str(d) for d in districts_of_policy_implementation])

path = "../../../../data/" +  string_is_for_1pm + "_simulations/"
basecase_subdir = pio.get_subdirs(path + string_is_for_1pm + "_basecase/")
comparison_subdir = pio.get_subdirs(path + string_is_for_1pm + "_policy_in_zone_2")

result_path_basecase_mean = "results/" + string_is_for_1pm + "_basecase_mean_links.geojson"
result_path_comparison_mean = "results/gdf_" + string_is_for_1pm + "_policy_in_" + string_district_of_interest + ".geojson"
result_path_difference = "results/gdf_" + string_is_for_1pm + "_difference.geojson"
result_path_average_mode_stats = "results/" + string_is_for_1pm + "_mean_mode_stats.csv"

compute_comparison_with_basecase = True

In [2]:
random_seed_2_df_basecase_output_links = pio.create_dic_seed_2_output_links(subdir=basecase_subdir)
random_seed_2_df_basecase_eqasim_trips = pio.create_dic_seed_2_eqasim_trips(subdir=basecase_subdir)
basecase_output_links_gdfs = list(random_seed_2_df_basecase_output_links.values())
gdf_basecase_mean = pio.compute_average_or_median_geodataframe(geodataframes=basecase_output_links_gdfs, column_name="vol_car", is_mean=True)
gdf_basecase_mean = gdf_basecase_mean.rename(columns={"osm:way:highway": "highway"})

# For the basecase mean, we remove duplicate entries.

In [3]:
def find_duplicate_edges_in_gdf(gdf):
    edge_count = defaultdict(list)
    for idx, row in gdf.iterrows():
        # Keep the edge direction by using the tuple without sorting
        edge = (row['from_node'], row['to_node'])
        edge_count[edge].append(idx)
    
    # Filter to include only edges that appear more than once
    duplicates = {edge: indices for edge, indices in edge_count.items() if len(indices) > 1}
    return duplicates

def summarize_duplicate_edges(gdf):
    if 'vol_car' not in gdf.columns:
        print("'vol_car' column does not exist in the dataframe")
        return gdf

    gdf['edge_id'] = gdf.apply(lambda row: (row['from_node'], row['to_node']), axis=1)
    grouped = gdf.groupby('edge_id')
    
    def aggregate_edges(group):
        non_zero_vol = group[group['vol_car'] != 0]
        if len(non_zero_vol) > 1:
            # If there are multiple non-zero entries, take the one with the highest vol_car
            combined = non_zero_vol.loc[non_zero_vol['vol_car'].idxmax()].copy()
        elif not non_zero_vol.empty:
            combined = non_zero_vol.iloc[0].copy()
        else:
            combined = group.iloc[0].copy()
        
        # We're no longer summing vol_car, just keeping the value from the selected row
        combined['original_directions'] = list(group[['from_node', 'to_node']].itertuples(index=False, name=None))
        return combined
    
    summarized_gdf = grouped.apply(aggregate_edges)
    summarized_gdf = summarized_gdf.reset_index(drop=True)
    summarized_gdf = summarized_gdf.drop(columns=['edge_id'])
    return summarized_gdf

remaining_duplicates = find_duplicate_edges_in_gdf(gdf_basecase_mean)
print(f"Number of duplicate edges: {len(remaining_duplicates)}")

links_without_duplicates = summarize_duplicate_edges(gdf_basecase_mean)
print(f"Number of remaining duplicate edges after summarizing: {len(find_duplicate_edges_in_gdf(links_without_duplicates))}")

Number of duplicate edges: 74
Number of remaining duplicate edges after summarizing: 0


In [4]:
def identify_summarized_entries_detailed(original_gdf, summarized_gdf):
    original_gdf['edge_id'] = original_gdf['from_node'].astype(str) + '_' + original_gdf['to_node'].astype(str)
    summarized_gdf['edge_id'] = summarized_gdf['from_node'].astype(str) + '_' + summarized_gdf['to_node'].astype(str)
    
    original_counts = original_gdf['edge_id'].value_counts()
    summarized_entries = summarized_gdf[summarized_gdf['edge_id'].isin(original_counts[original_counts > 1].index)]
    
    detailed_entries = []
    both_zero = []
    both_nonzero = []
    one_zero_one_nonzero = []
    
    for _, summarized_row in summarized_entries.iterrows():
        edge_id = summarized_row['edge_id']
        original_rows = original_gdf[original_gdf['edge_id'] == edge_id]
        
        vol_car_values = original_rows['vol_car'].values
        entry = {
            'summarized': summarized_row,
            'original': original_rows,
            'count': len(original_rows)
        }
        
        if all(vol_car == 0 for vol_car in vol_car_values):
            both_zero.append(entry)
        elif all(vol_car != 0 for vol_car in vol_car_values):
            both_nonzero.append(entry)
        else:
            one_zero_one_nonzero.append(entry)
        
        detailed_entries.append(entry)
    
    return detailed_entries, both_zero, both_nonzero, one_zero_one_nonzero

detailed_entries, both_zero, both_nonzero, one_zero_one_nonzero = identify_summarized_entries_detailed(gdf_basecase_mean, links_without_duplicates)

print(f"Number of summarized entries: {len(detailed_entries)}")
print(f"Entries where both 'vol_car' are zero: {len(both_zero)}")
print(f"Entries where both 'vol_car' are non-zero: {len(both_nonzero)}")
print(f"Entries where one 'vol_car' is zero and one is non-zero: {len(one_zero_one_nonzero)}")

def print_examples(category, entries, num_examples=2):
    print(f"\n{category} (showing {min(num_examples, len(entries))} examples):")
    for i, entry in enumerate(entries[:num_examples]):
        print(f"\nExample {i+1}:")
        print("Summarized row:")
        print(entry['summarized'])
        print("\nOriginal rows:")
        print(entry['original'])
        print(f"Count: {entry['count']}")
        print("-" * 50)

print_examples("Both 'vol_car' are zero", both_zero)
print_examples("Both 'vol_car' are non-zero", both_nonzero)
print_examples("One 'vol_car' is zero and one is non-zero", one_zero_one_nonzero)

Number of summarized entries: 74
Entries where both 'vol_car' are zero: 25
Entries where both 'vol_car' are non-zero: 9
Entries where one 'vol_car' is zero and one is non-zero: 40

Both 'vol_car' are zero (showing 2 examples):

Example 1:
Summarized row:
link                                                                    412904
from_node                                                            112017915
to_node                                                              112017915
length                                                               39.819261
freespeed                                                             8.333333
capacity                                                                 480.0
lanes                                                                      1.0
modes                                                        car,car_passenger
vol_car                                                                    0.0
osm:relation:route_master         

## Fill Nan Values

The values freespeed and highway, which we will use later, have nan values. We need to approximate it.

In [5]:
# Filter entries where 'highway' is NaN
highway_nan_entries = links_without_duplicates[links_without_duplicates['highway'].isna()]

# Get the distribution of the column 'modes' for these entries
modes_distribution = highway_nan_entries['modes'].value_counts()

# Print the distribution
print("Distribution of 'modes' where 'highway' is NaN:")
print(modes_distribution)

# Check if all 'modes' are "pt,rail,train"
all_modes_are_pt_rail_train = (highway_nan_entries['modes'] == "pt,rail,train").all()
print("\nIs it true that when 'highway' is NaN, 'modes' is always 'pt,rail,train'?")
print(all_modes_are_pt_rail_train)

# For those entries where "highway" is currently NaN, set "highway" to "pt"
links_without_duplicates.loc[links_without_duplicates['highway'].isna(), 'highway'] = 'pt'

Distribution of 'modes' where 'highway' is NaN:
pt,rail,train                            971
artificial,stopFacilityLink,subway       641
artificial,subway                        609
rail                                     347
artificial,stopFacilityLink,tram         112
artificial,tram                          109
artificial,bus                            53
pt,subway                                  9
artificial,funicular,stopFacilityLink      4
artificial,bus,stopFacilityLink            3
artificial,rail                            2
artificial,funicular                       2
Name: modes, dtype: int64

Is it true that when 'highway' is NaN, 'modes' is always 'pt,rail,train'?
False


In [6]:
# Calculate the average freespeed for each highway type
average_freespeed_by_highway = links_without_duplicates.groupby('highway')['freespeed'].mean()

# Fill NaN freespeed values with the average freespeed of their respective highway type
for highway_type, avg_freespeed in average_freespeed_by_highway.items():
    print(f"Highway type: {highway_type}")
    print(f"Average freespeed: {avg_freespeed}")
    mask = (links_without_duplicates['freespeed'].isna()) & (links_without_duplicates['highway'] == highway_type)
    links_without_duplicates.loc[mask, 'freespeed'] = avg_freespeed

# Check if there are any remaining NaN values in freespeed
remaining_nan = links_without_duplicates['freespeed'].isna().sum()
print(f"\nRemaining NaN values in freespeed: {remaining_nan}")

Highway type: construction
Average freespeed: 13.194444444444445
Highway type: living_street
Average freespeed: 5.100940052396363
Highway type: motorway
Average freespeed: 22.290688575899843
Highway type: motorway_link
Average freespeed: 15.2689313517339
Highway type: pedestrian
Average freespeed: 8.955938697318008
Highway type: primary
Average freespeed: 10.505286174793174
Highway type: primary_link
Average freespeed: 9.945208970438328
Highway type: pt
Average freespeed: nan
Highway type: residential
Average freespeed: 8.161352865144334
Highway type: secondary
Average freespeed: 9.1430231943784
Highway type: secondary_link
Average freespeed: 9.245439469320067
Highway type: service
Average freespeed: 10.0664767331434
Highway type: tertiary
Average freespeed: 8.513650453028136
Highway type: tertiary_link
Average freespeed: 9.146341463414634
Highway type: trunk
Average freespeed: 19.382371198013654
Highway type: trunk_link
Average freespeed: 12.022249522249522
Highway type: unclassified


In [7]:
# Check for NaN values in all columns of links_without_duplicates
nan_values = links_without_duplicates.isna().sum()

# Print the columns with their respective count of NaN values
print("Count of NaN values in each column:")
print(nan_values)

Count of NaN values in each column:
link                             0
from_node                        0
to_node                          0
length                           0
freespeed                        0
capacity                         0
lanes                            0
modes                            0
vol_car                          0
osm:relation:route_master    31134
osm:way:vehicle              31124
osm:way:traffic_calming      31060
osm:way:junction             30514
osm:way:motorcycle           31110
isUrban                       3203
osm:way:lanes                19097
osm:way:psv                  30664
osm:way:service              30677
osm:way:id                    1535
osm:way:access               30457
osm:way:oneway               12418
highway                          0
osm:relation:route           14082
osm:way:railway              29813
osm:way:name                  3368
storageCapacityUsedInQsim    29774
osm:way:tunnel               30278
geometry           

In [8]:
links_without_duplicates.head()

,link,from_node,to_node,length,freespeed,capacity,lanes,modes,vol_car,osm:relation:route_master,...,osm:way:oneway,highway,osm:relation:route,osm:way:railway,osm:way:name,storageCapacityUsedInQsim,osm:way:tunnel,geometry,original_directions,edge_id
0,324663,1000258241,8880659948,20.037097,8.333333,7999.2,1.0,"pt,rail,train",0.0,NaN,...,NaN,pt,"railway,train",rail,Lignes de Paris-Est à Strasbourg et Mulhouse,0.032056,yes,"LINESTRING (2.36296 48.88421, 2.36306 48.88437)","[(1000258241, 8880659948)]",1000258241_8880659948
1,261428,1000258241,8880659951,9.776025,8.333333,7999.2,1.0,"pt,rail,train",0.0,NaN,...,NaN,pt,"railway,train",rail,Lignes de Paris-Est à Strasbourg et Mulhouse,0.015640,NaN,"LINESTRING (2.36296 48.88421, 2.36291 48.88413)","[(1000258241, 8880659951)]",1000258241_8880659951
2,609662,1000258242,5646436942,8.333333,8.333333,7999.2,1.0,"pt,rail,train",0.0,NaN,...,NaN,pt,train,rail,Paris Gare de l'Est,NaN,NaN,"LINESTRING (2.36135 48.88099, 2.36136 48.88105)","[(1000258242, 5646436942)]",1000258242_5646436942
3,312238,1000258242,6556226602,26.739746,8.333333,7999.2,1.0,"pt,rail,train",0.0,NaN,...,NaN,pt,train,rail,Paris Gare de l'Est,0.042779,NaN,"LINESTRING (2.36135 48.88099, 2.36131 48.88075)","[(1000258242, 6556226602)]",1000258242_6556226602
4,69,1000258245,1000258255,36.086118,8.333333,7999.2,1.0,"pt,rail,train",0.0,NaN,...,NaN,pt,train,rail,Paris Gare de l'Est,0.057732,NaN,"LINESTRING (2.36170 48.88192, 2.36154 48.88161)","[(1000258245, 1000258255)]",1000258245_1000258255


In [9]:
gdf_to_save = links_without_duplicates.copy()
columns_to_drop = [col for col in gdf_to_save.columns if col.startswith('osm:')]
columns_to_drop.append('original_directions')
gdf_to_save.drop(columns=columns_to_drop, inplace=True)
gdf_to_save = gdf_to_save.set_crs("EPSG:4326", allow_override=True)
gdf_to_save.to_file(result_path_basecase_mean, driver='GeoJSON')

In [14]:
def calculate_avg_mode_stats(single_mode_stats_list:list):
    mode_stats_list = []

    for df in single_mode_stats_list:
        mode_stats = df.groupby('mode').agg({
            'travel_time': 'mean',
            'routed_distance': 'mean'
        }).reset_index()
        mode_stats.columns = ['mode', 'avg_travel_time', 'avg_routed_distance']
        mode_stats_list.append(mode_stats)
        
    # Concatenate all mode_stats dataframes
    all_mode_stats = pd.concat(mode_stats_list, ignore_index=True)

    # Calculate the average across all seeds
    average_mode_stats = all_mode_stats.groupby('mode').agg({
        'avg_travel_time': 'mean',
        'avg_routed_distance': 'mean'
    }).reset_index()
    average_mode_stats.columns = ['mode', 'avg_total_travel_time', 'avg_total_routed_distance']
    df_average_mode_stats = pd.DataFrame(average_mode_stats)
    return df_average_mode_stats

df_average_mode_stats = calculate_avg_mode_stats(random_seed_2_df_basecase_eqasim_trips.values())
df_average_mode_stats.to_csv(result_path_average_mode_stats, index=False)

In [15]:
df_average_mode_stats

,mode,avg_total_travel_time,avg_total_routed_distance
0,bike,1187.134382,3681.674165
1,car,941.046459,4835.211251
2,car_passenger,423.112003,4378.124317
3,outside,0.788800,1057.635258
4,pt,1602.507719,5467.415175
5,walk,1007.712862,1209.811859


In [4]:
# if compute_comparison_with_basecase:
#     random_seed_2_df_comparison = pio.create_dic_seed_2_output_links(subdir = comparison_subdir)
#     geodataframes_comparison = list(random_seed_2_df_comparison.values())
#     gdf_comparison_mean = pio.compute_average_or_median_geodataframe(geodataframes=geodataframes_comparison, column_name="vol_car", is_mean=True)
#     gdf_comparison_mean_extended = pio.extend_geodataframe(gdf_base = gdf_basecase_mean, gdf_to_extend=gdf_comparison_mean, column_to_extend='highway', new_column_name='highway')
#     gdf_basecase_without_unnecessary_columns = pio.remove_columns(gdf_with_correct_columns=gdf_comparison_mean_extended, gdf_to_be_adapted=gdf_basecase_mean)
#     gdf_basecase_difference = pio.compute_difference_geodataframe(gdf_to_substract_from=gdf_comparison_mean_extended, gdf_to_substract=gdf_basecase_without_unnecessary_columns, column_name= 'vol_car')
#     gdf_comparison_mean_extended.to_file(result_path_comparison_mean, driver='GeoJSON')
#     gdf_basecase_difference.to_file(result_path_difference, driver='GeoJSON')